In [9]:
import cv2
import os
import numpy as np
import onnx

In [17]:
# 경로 설정
base_path  = os.getcwd()
model_path = os.path.join(base_path, "models" ,"yolov3_tiny")
onnx_path  = os.path.join(model_path, "yolov3-tiny.onnx")
name_path  = os.path.join(model_path, "coco.names")
img_dir_path   = os.path.join(base_path, "data-files", "images")

img_size = 416

In [18]:
print("베이스 경로:", base_path, os.path.exists(base_path))
print("모델 경로:", model_path, os.path.exists(model_path))
print("ONNX 경로:", onnx_path, os.path.exists(onnx_path))
print("클래스 경로:", name_path, os.path.exists(name_path))
print("이미지 경로:", os.path.join(img_dir_path, "1.jpg"), os.path.exists(os.path.join(img_dir_path, "1.jpg")))

print("존재 여부:", os.path.exists(onnx_path))
print("파일 크기 (MB):", os.path.getsize(onnx_path)/1024/1024)

model = onnx.load(onnx_path)  # 모델 로드
onnx.checker.check_model(model)
print("ONNX 모델 유효성 확인 완료")

베이스 경로: c:\Users\human\Desktop\human\Workspace\dl-bagic True
모델 경로: c:\Users\human\Desktop\human\Workspace\dl-bagic\models\yolov3_tiny True
ONNX 경로: c:\Users\human\Desktop\human\Workspace\dl-bagic\models\yolov3_tiny\yolov3-tiny.onnx True
클래스 경로: c:\Users\human\Desktop\human\Workspace\dl-bagic\models\yolov3_tiny\coco.names True
이미지 경로: c:\Users\human\Desktop\human\Workspace\dl-bagic\data-files\images\1.jpg True
존재 여부: True
파일 크기 (MB): 33.767255783081055
ONNX 모델 유효성 확인 완료


In [24]:
def load_yolo():
    net = cv2.dnn.readNetFromONNX(onnx_path)

    with open(name_path) as f:
        classes = [c.strip() for c in f]

    return net, classes

def load_img(img_name):
    img_path   = os.path.join(img_dir_path, img_name)
    img = cv2.imread(img_path)
    return img

def detect_objects(net, classes, img):
    h, w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(img, 1/255.0, (img_size, img_size), swapRB=True, crop=False)
    net.setInput(blob)
    
    outs = net.forward()
    outs = outs.reshape(-1, outs.shape[-1])  # (25200, 85)

    conf_th, nms_th = 0.5, 0.4
    boxes, confs, ids = [], [], []

    for x, y, bw, bh, obj, *cls in outs:
        obj = float(obj)
        if obj < conf_th: continue
        score = max(cls)
        cid = np.argmax(cls)
        if score > conf_th:
            # 좌표 변환 (중심 좌표 -> 왼쪽 위 좌표)
            px, py, pw, ph = x * w, y * h, bw * w, bh * h
            x1 = int(px - pw / 2)
            y1 = int(py - ph / 2)
            boxes.append([x1, y1, int(pw), int(ph)])
            confs.append(float(score))
            ids.append(int(cid))

    idxs = cv2.dnn.NMSBoxes(boxes, confs, conf_th, nms_th)
    for i in idxs.flatten():
        x,y,bw,bh = boxes[i]
        label = f"{classes[ids[i]]}: {confs[i]:.2f}"
        cv2.rectangle(img, (x, y), (x+bw, y+bh), (0,255,0), 2)
        cv2.putText(img, label, (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    return img




In [26]:
net, classes = load_yolo()

img = load_img("2.jpg")

result = detect_objects(net, classes, img)

cv2.imshow("YOLOv3-Tiny Detection", result)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [30]:
from ultralytics import YOLO

# 사전 학습된 YOLOv8 모델 불러오기 (작고 빠른 nano 버전)
model = YOLO("yolov8s.pt")  # 또는 yolov8s.pt (조금 더 정확하지만 약간 느림)

# 이미지 로드 및 탐지
img_path   = os.path.join(img_dir_path, "2.jpg")
results = model(img_path)

# 결과 표시 (Ultralytics 내장 뷰어)
res = results[0]

# 탐지 박스가 그려진 이미지 (NumPy 배열)
res_img = res.plot()

# OpenCV로 출력
cv2.imshow("YOLOv8 Detection", res_img)
cv2.waitKey(0)
cv2.destroyAllWindows()



image 1/1 c:\Users\human\Desktop\human\Workspace\dl-bagic\data-files\images\2.jpg: 448x640 1 dog, 198.3ms
Speed: 4.6ms preprocess, 198.3ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)
